# Create Templeton Awards from WordPress REST API

Creates John Templeton Foundation awards. ~5,956 grants from
templeton.org via the WP REST `/wp-json/wp/v2/grants` endpoint, which
exposes a custom ACF block with grant_amount (USD), start/end dates,
grantee organisation, and project leader.

**Prerequisites:**
- Run `scripts/local/templeton_to_s3.py` to download and upload the data.

**Data source:** templeton.org WP REST API (ACF custom fields)
**S3 location:** `s3a://openalex-ingest/awards/templeton/templeton_projects.parquet`

**Templeton funder (OpenAlex):**
- funder_id: 4320306193
- ROR: https://ror.org/035tnyy05
- DOI: 10.13039/100000925
- display_name: "John Templeton Foundation"

**Schema notes:**
- `grant_amount_raw` is a string in the source (e.g. "245005"); we
  TRY_CAST to DOUBLE in the transformation and document USD currency.
- `grant_project_leader` may contain multiple PIs separated by `;`
  (e.g. "Stephen Bullivant; Victoria Lorrimar"); we keep the raw text
  in lead_investigator family/given fields and split-on-first-PI is
  left for a future enrichment.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.templeton_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/templeton/templeton_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.templeton_raw;

## Step 1.5: Inspect raw data + verify amount/currency (per how-to Step 1.6)

In [ ]:
%sql
DESCRIBE openalex.awards.templeton_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.templeton_raw LIMIT 5;

In [ ]:
%sql
-- Confirm amount is parseable to a sensible distribution
SELECT
  COUNT(*) AS rows,
  COUNT(grant_amount_raw) AS has_amount_raw,
  COUNT(TRY_CAST(grant_amount_raw AS DOUBLE)) AS parseable_amount,
  ROUND(MIN(TRY_CAST(grant_amount_raw AS DOUBLE)), 0) AS min_amt,
  ROUND(MAX(TRY_CAST(grant_amount_raw AS DOUBLE)), 0) AS max_amt,
  ROUND(AVG(TRY_CAST(grant_amount_raw AS DOUBLE)), 0) AS avg_amt
FROM openalex.awards.templeton_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.templeton_awards
USING delta
AS
WITH templeton_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306193
),
src AS (
    SELECT
        wp_post_id,
        slug,
        url,
        COALESCE(NULLIF(title, ''), grant_web_title) AS display_name,
        COALESCE(grant_max_content, grant_content) AS description,
        grant_id AS funder_award_id,
        TRY_CAST(grant_amount_raw AS DOUBLE) AS amount,
        TRY_TO_DATE(SUBSTRING(grant_start_date, 1, 10), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(SUBSTRING(grant_end_date, 1, 10), 'yyyy-MM-dd') AS end_date,
        grant_grantee,
        grant_project_leader
    FROM openalex.awards.templeton_raw
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(s.funder_award_id)))) % 9000000000 AS id,
    s.display_name,
    s.description,
    f.funder_id,
    s.funder_award_id,
    s.amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    CAST(NULL AS STRING) AS funder_scheme,
    'templeton_wp' AS provenance,
    s.start_date,
    s.end_date,
    YEAR(s.start_date) AS start_year,
    YEAR(s.end_date) AS end_year,
    -- Project leader → lead_investigator. Multi-PI strings are kept in family_name verbatim
    -- so they're searchable; downstream enrichment can split on ';'.
    struct(
        CAST(NULL AS STRING) AS given_name,
        s.grant_project_leader AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            s.grant_grantee AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    s.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(s.funder_award_id)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM src s
CROSS JOIN templeton_funder f
WHERE s.funder_award_id IS NOT NULL AND TRIM(s.funder_award_id) != '';

## Step 3: Insert into openalex_awards_raw at priority 39

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'templeton_wp' AND priority = 39;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    39 AS priority
FROM openalex.awards.templeton_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_templeton_awards FROM openalex.awards.templeton_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.templeton_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_award_id, amount, currency, start_date, end_date,
       lead_investigator.affiliation.name AS grantee
FROM openalex.awards.templeton_awards LIMIT 10;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.templeton_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) FROM openalex.awards.templeton_awards GROUP BY funder.display_name;